In [ ]:
# instalação da biblioteca do google de ia generativa
!pip install -U google-genai


In [2]:
import os  # biblioteca para acessar as variaveis e ambiente
from google.colab import userdata # importa o userdata para carregar variaveis


In [3]:
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
# importa json para transformar listas e dicionarios em texto json
import json
# importa o cliente da API Gemini
from google import genai
# importa tipos de configuração da biblioteca Gemini
from google.genai import types


# Catalogo de produtos do petshop

Catalogo_petshop = [
    {
        'nome': 'Ração Golden Fórmula Cães Adultos 15 Kg',
        'categoria': 'Ração',
        'porte': 'Adulto',
        'marca':'Golden',
        'preco': 149.90
    },

    {
        'nome': 'Ração Premier Fórmula Cães Filhotes 10 Kg',
        'categoria': 'Ração',
        'porte': 'filhote',
        'marca': 'Premier',
        'preco': 189.90
    },

    {
        'nome': 'Ração Whiskas Carne Gatos Adultos 10 Kg',
        'categoria': 'Ração para gato',
        'porte': 'adulto',
        'marca': 'Whiskas',
        'preco': 129.90
    },

    {
        'nome': 'Ração Royal Canil  Gatos Castrados 7.5 Kg',
        'categoria': 'Ração para gato',
        'porte': 'Adulto',
        'marca': 'Royal Canin',
        'preco': 239.90
    }

]


# Cria o cliente da API Gemini

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])


# Cria função para extrair_preco
def extrair_preco_maximo(pergunta):
  """
  Tenta encontrar um valor numérico na pergunta para usar como filtro de preço
  Exemplo: ração até 150 reais -> 150.0
  """
  palavras = pergunta.replace(",", ".").split()

  for palavra in palavras:
    try:
      return float(palavra)
    except ValueError:
      continue
  return None


# Cria função para fazer a busca no catalogo com regras mais inteligentes

def buscar_produtos(pergunta):
  """
  Busca produtos no catalogo com regras mais inteligentes
  """

  pergunta = pergunta.lower().strip()

  #Se a pessoa pedir catalogo completo ou tipos de ração retorna tudo

  gatilhos_catalogo =[
      "catalogo","catálogo","produtos","mostrar tudo","mostre tudo",
      "todas as rações","todas as racoes","tipos de ração","tipos de racao",
      "quais rações","quais racoes","quais produtos"
  ]
  if any(gatilho in pergunta for gatilho in gatilhos_catalogo):

    return Catalogo_petshop

  resultados =[]


  # Filtros identificados na pergunta

  quer_cachorro  = any (p in pergunta for p in ['cachorro','cão', 'cao','cães','caes'])
  quer_gato = "gato" in pergunta or "gatos" in pergunta
  quer_filhote = "filhote" in pergunta or "filhotes" in pergunta
  quer_adulto = "adulto" in pergunta or "adultos" in pergunta

  preco_maximo = None

  if "até" in pergunta or 'ate' in pergunta or 'menos de' in pergunta:
    preco_maximo = extrair_preco_maximo(pergunta)

  # palavras comuns que nao ajudam na busca

  palavras_ignoradas ={
      "de","do","da", "das","dos","para","com","sem","e",
      "ou","a","o","as","os","um","uma","quero","qual","quais",
      "tem","temos","vocês","voces","ração","rações","racoes"
  }

  palavras_busca =[p for p in pergunta.split() if p not in palavras_ignoradas]

  for produto in Catalogo_petshop:
    # Começa assumindo que o produto atende

    atende = True
    #Filtro por tipo de animal

    if quer_cachorro and "cachorro" not in produto["categoria"]:
      atende = False
    elif quer_gato and "gato" not in produto["categoria"]:
      atende = False

    # Filtro por porte

    if quer_filhote and produto["porte"] != "filhote":
      atende = False
    elif quer_adulto and produto["porte"] != "adulto":
      atende = False

    # Filtro por preço

    if preco_maximo is not None and produto["preco"]> preco_maximo:
      atende = False

    # Busca por marcas ou palavras relevantes

    texto_produto = f"{produto['nome']} {produto['categoria']} {produto['porte']} {produto['marca']}".lower()
    if not (quer_cachorro or quer_gato or quer_filhote or quer_adulto or preco_maximo is not None):
      if palavras_busca:

       if not any(palavra in texto_produto for palavra in palavras_busca):
        atende = False

    if atende:
      resultados.append(produto)

  return resultados



# função para responder o cliente

def responder_cliente(pergunta):
  """
  Gera a resposta do agente com base nos produtos encontrados.
  """

  produtos_encontrados = buscar_produtos(pergunta)
  contexto_catalogo = json.dumps(produtos_encontrados,ensure_ascii=False, indent=2)

  prompt = f"""
  Você é um agente virtual de um petshop.

  Regras:

  - Responda somente com base nos produtos encontrados no catálogo.
  - Não invente produtos ou preços.
  - Se não encontrar produtos, diga claramente que não encontrou no sistema.
  - Se o cliente pedir o catálogo ou tipos de ração, apresente os produtos disponíveis
  - Responda em português do Brasil.
  - Seja educado e objetivo

Pergunta do cliente:
{pergunta}

Produtos encontrados:

{contexto_catalogo}
"""

  # modelo

  response = client.models.generate_content(
    model = "gemini-2.5-flash-lite",
    contents = prompt,
    config = types.GenerateContentConfig(
    system_instruction = "Você é um atendente virtual especializado em produtos de petshop")

    )
  return response.text


# loop principal

while True:

   pergunta = input("\n Cliente: ")

   if pergunta.lower() in ["sair","exit","quit"]:
      print("Encerrando atendimento")

      break
   resposta = responder_cliente(pergunta)
   print("\n Agente: ",resposta)
